# 05 — User audio profiles (Phase 2, Step 4)

Для каждого пользователя считаем `a_bar_u` — mean аудиоэмбеддингов треков из его train-истории (listen+, `min_pop>=5`, только items с `audio_valid`).

**Вход:** `artifacts/audio/embeddings.npy` (276,306 × 128), `artifacts/gsasrec/item_id_to_idx.pkl`, raw yambda-50m с HF.

**Выход:** `artifacts/audio/user_profiles.npy` `[n_users, 128]`, `uid_to_row.pkl`, `user_audio_valid.npy`.

In [1]:
# Colab bootstrap (раскомментировать в Colab):
from google.colab import userdata


token = userdata.get('git')
!git clone -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
%cd music-recommendations
!pip install -q datasets pyarrow numpy pandas

Cloning into 'music-recommendations'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 181 (delta 66), reused 155 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 705.78 KiB | 14.70 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/music-recommendations


In [2]:
import os, sys, pickle
from pathlib import Path
import numpy as np


PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

project root: /content/music-recommendations


In [3]:
(Path(PROJECT_ROOT) / 'artifacts' / 'gsasrec').mkdir(parents=True, exist_ok=True)

In [4]:
from src.data.yambda_loader import (
    load_yambda, filter_listens, filter_min_popularity, apply_item_remap,
)
from src.data.splits import global_temporal_split, SplitConfig

HF_CACHE = os.environ.get('HF_DATASETS_CACHE', None)
raw = load_yambda('50m', cache_dir=HF_CACHE)['interactions']
print('raw events:', len(raw))

df = filter_listens(raw)
df = filter_min_popularity(df, min_count=5)

with open(PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)
print(f'item_id_to_idx: {len(item_id_to_idx):,} items')

# Sanity: пайплайн post-filter должен давать те же items, что в pkl. Если когда-нибудь
# поедет — упадём на map (Nan) и сразу заметим.
df = apply_item_remap(df, item_id_to_idx)
assert df['item_idx'].isna().sum() == 0, 'item_id_to_idx out of sync with current filter pipeline'

train, val, test = global_temporal_split(df, SplitConfig())
print(f'train: {len(train):,} events, {train.uid.nunique():,} users')

README.md: 0.00B [00:00, ?B/s]

flat/50m/multi_event.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47790449 [00:00<?, ? examples/s]

raw events: 47790449
item_id_to_idx: 276,305 items
train: 28,524,373 events, 9,194 users


In [6]:
(Path(PROJECT_ROOT) / 'artifacts' / 'audio').mkdir(parents=True, exist_ok=True)

In [7]:
EMB_PATH = PROJECT_ROOT / 'artifacts' / 'audio' / 'embeddings.npy'
item_embeddings = np.load(EMB_PATH)
print('embeddings:', item_embeddings.shape, item_embeddings.dtype)

audio_valid_items = np.linalg.norm(item_embeddings, axis=1) > 0
print(f'audio_valid items: {int(audio_valid_items[1:].sum()):,} / {item_embeddings.shape[0]-1:,} '
      f'({100*audio_valid_items[1:].mean():.2f}%)')

embeddings: (276306, 128) float32
audio_valid items: 264,840 / 276,305 (95.85%)


In [8]:
from src.data.audio_embeddings import build_user_audio_profiles

profiles, uid_to_row, user_audio_valid = build_user_audio_profiles(train, item_embeddings)
print('profiles shape:', profiles.shape)
print('users with audio:', int(user_audio_valid.sum()), '/', len(user_audio_valid))

[audio] user profiles: 9194 users, with audio: 9190 (99.96%), norm mean (valid): 17.803
profiles shape: (9194, 128)
users with audio: 9190 / 9194


In [9]:
OUT_DIR = PROJECT_ROOT / 'artifacts' / 'audio'
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / 'user_profiles.npy', profiles)
np.save(OUT_DIR / 'user_audio_valid.npy', user_audio_valid)
with open(OUT_DIR / 'uid_to_row.pkl', 'wb') as f:
    pickle.dump(uid_to_row, f)

print('saved:')
print(f"  user_profiles.npy     {profiles.nbytes / 2**20:.2f} MB")
print(f"  user_audio_valid.npy  {user_audio_valid.nbytes / 2**10:.2f} KB")
print(f"  uid_to_row.pkl        {len(uid_to_row):,} entries")

saved:
  user_profiles.npy     4.49 MB
  user_audio_valid.npy  8.98 KB
  uid_to_row.pkl        9,194 entries


In [10]:
# Sanity на одного юзера: вручную пересчитать профиль и сравнить.
import random
rng = random.Random(0)
uid = rng.choice([u for u, v in zip(uid_to_row, user_audio_valid) if v])
items = train.loc[train['uid'] == uid, 'item_idx'].to_numpy()
valid_items = items[audio_valid_items[items]]
expected = item_embeddings[valid_items].mean(axis=0)
got = profiles[uid_to_row[uid]]
diff = float(np.abs(expected - got).max())
print(f'uid={uid}, history={len(items)}, valid={len(valid_items)}, max|diff|={diff:.2e}')
assert diff < 1e-5, 'profile mismatch'
print('sanity OK')

uid=686900, history=8307, valid=8177, max|diff|=0.00e+00
sanity OK
